In [12]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import laspy
import random

def process_las_files_split3(
    directory, out_root, chunk_size=100000, tile_name="tile",
    train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42
):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios müssen zusammen 1.0 ergeben"

    files = [os.path.join(directory, f) for f in os.listdir(directory)
             if f.endswith(".las") or f.endswith(".laz")]
    header = []
    chunk_data_buffer = []
    chunk_counter = 0
    random.seed(seed)

    for file_idx, file in enumerate(tqdm(files, desc="Lade .las-Dateien")):
        try:
            las = laspy.read(file)
            dims = list(las.points.point_format.dimension_names)
            scale = las.header.scale
            offset = las.header.offset
            if not header:
                header = dims
            total_points = len(las.points)
        except Exception as e:
            print(f"\nFehler beim Lesen von {file}: {e}")
            continue

        start = 0
        while start < total_points:
            end = min(start + chunk_size, total_points)
            chunk_data = []
            valid_chunk = True
            for d in dims:
                try:
                    data_array = np.array(las.points[d][start:end], dtype=float)
                    if d in ["X", "Y", "Z"]:
                        idx = ["X", "Y", "Z"].index(d)
                        data_array = data_array * scale[idx] + offset[idx]
                    if data_array.size != (end - start):
                        print(f"\nWarnung: Inkonsistente Punktzahl in Chunk {chunk_counter}")
                        valid_chunk = False
                        break
                    chunk_data.append(data_array)
                except Exception as e:
                    print(f"\nFehler bei Dimension '{d}': {e}")
                    valid_chunk = False
                    break

            if not valid_chunk or len(chunk_data) != len(dims):
                start = end
                continue

            chunk = np.vstack(chunk_data).T
            chunk_df = pd.DataFrame(chunk, columns=header)

            feature_cols = ["X", "Y", "Z", "intensity", "return_number", "number_of_returns"]
            label_col = "classification"

            if not all(col in chunk_df.columns for col in feature_cols + [label_col]):
                print(f"\n❌ Chunk {chunk_counter} übersprungen – fehlende Spalten.")
                start = end
                continue

            features = chunk_df[feature_cols].to_numpy()
            labels = chunk_df[label_col].to_numpy().astype(np.int32)

            chunk_data_buffer.append((features, labels, f"{tile_name}_chunk{chunk_counter}"))
            chunk_counter += 1

            start = end

    if not chunk_data_buffer:
        print("\n⚠️ Keine gültigen Chunks extrahiert.")
        return

    # Shuffle and split
    random.shuffle(chunk_data_buffer)
    n_total = len(chunk_data_buffer)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)
    n_test = n_total - n_train - n_val

    splits = {
        "train": chunk_data_buffer[:n_train],
        "val": chunk_data_buffer[n_train:n_train + n_val],
        "test": chunk_data_buffer[n_train + n_val:]
    }

    for split_name, chunks in splits.items():
        split_dir = os.path.join(out_root, split_name)
        os.makedirs(split_dir, exist_ok=True)
        for features, labels, name in chunks:
            points_path = os.path.join(split_dir, f"{name}_points.npy")
            labels_path = os.path.join(split_dir, f"{name}_labels.npy")
            np.save(points_path, features)
            np.save(labels_path, labels)

    print(f"\n✅ Aufteilung abgeschlossen:")
    print(f"  🔹 Train: {n_train} Chunks")
    print(f"  🔸 Val:   {n_val} Chunks")
    print(f"  🔻 Test:  {n_test} Chunks")
    print(f"  📂 Gespeichert in: '{out_root}/{{train,val,test}}'")

In [13]:
process_las_files_split3(
    directory="C:/Users/Nicola/Desktop/zurich_laz",
    out_root="./processed_chunks",
    chunk_size=100_000,
    tile_name="tileCH",
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15
)



Lade .las-Dateien: 100%|██████████| 20/20 [00:30<00:00,  1.50s/it]



✅ Aufteilung abgeschlossen:
  🔹 Train: 1200 Chunks
  🔸 Val:   257 Chunks
  🔻 Test:  258 Chunks
  📂 Gespeichert in: './processed_chunks/{train,val,test}'
